# Assignment 1

![](https://media.giphy.com/media/xT9C25UNTwfZuk85WP/giphy-downsized-large.gif)

Remember the rules of ~Fight~ Code Club:
1. ALWAYS DOCUMENT
2. Cite resources that you use (paste links)
3. Include the names people who you worked with
4. Be neat and organized

## Scrape and Clean data

Based on you proposal, scrape or collect your data:

1. One variable must be either: (40 pts)
    1. scraped from the web OR;
    2. collected from an API AND you must create one new variable that is "new" to the best of your knowledge (combination of other variables representing something new).

2. You must have at least 3 variables, but you may include as many as you want into you final dataset. Likely, you will want to include more to make graphs and regressions. (30pts)

3. You must *be able* to run a regression that makes some sense with this data (the regression doesn't have to be a complete model). Briefly describe one regression you would run with your variables. (**DO NOT** run a regression, yet). (15pts)

4. You must have one combined and cleaned dataset (15 pts)

You must submit one python notebook on how you scraped/gathered data from an api, and how you combined and cleaned you data. I should be able to run your code and reproduce your final data set.  

The other variables that you choose to include do not have to be collected by API or webscraped, but you do have to combine the files and clean the dataset with python.

Thus, you must submit:
- Your finalized data set (only one) (note: you may add more variables in the future).
- Your documented python notebook
- Any associated data files needed to produce the final dataset.

You will be evaluated on:
- Completeness of the data
- Quality of the code 
- The creativity of the new variable/webscraped data you gather

Be sure to upload ALL associated files for your code to run. I will run your code from beginning to end - make it easy for me to replicate your code.

In [63]:
import requests
import pandas as pd
import re
import time
from bs4 import BeautifulSoup

In [64]:
#Webscraping Michelin Website
base_url = "https://guide.michelin.com/us/en/new-york-state/new-york/restaurants" #storing base URL

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Connection": "keep-alive"
} #initalizing my headers

session = requests.Session() #opening browser session
all_rows = [] #creating empty list to store all restaurant data across all pages
page = 1 #starting on page 1
max_pages = 15 #safety stop so the loop cannot run forever

while page <= max_pages:
    # page 1 uses the base URL, later pages use /page/2, /page/3, etc.
    if page == 1:
        url = base_url
    else:
        url = f"{base_url}/page/{page}"

    print(f"\nScraping page {page}: {url}")

    response = session.get(url, headers=headers, timeout=30) #inital get request, setting headers, and setting a timeout of 30 seconds

    print("Status code:", response.status_code) #display status code
    print("Final URL:", response.url) #display the finale page URL
    print("Content-Type:", response.headers.get("Content-Type")) #display the type of content that has been returned
    print("Length:", len(response.text)) #display how much html I got back

    if len(response.text) == 0:
        print("Request returned empty HTML.") #Checking to see if HTML is empty so code doesn't crash later
        break

    # saves raw HTML if it is not empty so I do not lose a potential good pull
    with open(f"michelin_listing_raw_page_{page}.html", "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"Saved raw HTML as michelin_listing_raw_page_{page}.html")

    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.find_all("div", class_=lambda c: c and "js-restaurant__list_item" in c)
    print("Number of cards found:", len(cards)) #finding all the restaurant cards on the michelin website

    #if there are no cards found let me know so I can diagnose the issue
    #also stop the loop because it likely means there are no more pages with restaurant data
    if len(cards) == 0:
        print("No restaurant cards found.")
        break

    rows = [] #create empty list for just this page

    for card in cards:
        name_tag = card.find("h3") #find all the restaurant names
        a_tag = name_tag.find("a", href=True) if name_tag else None #also grabs the restaurnat's link if avaialable

        restaurant_name = a_tag.get_text(" ", strip=True) if a_tag else None #pulls out the restaurant name as text and remove surrounding spaces
        restaurant_url = ( #builds the full michelin URL by adding the base site and partial links
            "https://guide.michelin.com" + a_tag["href"]
            if a_tag and a_tag.get("href")
            else None
        )

        raw_card_text = card.get_text(" | ", strip=True) #extracts all visible text from the card
        cleaned_text = re.sub(r"\s+", " ", raw_card_text).strip() #cleans and seperates text to make it readable

        # location
        location_match = re.search(r"[A-Za-z'’&.\- ]+,\s*[A-Z]{2},\s*USA", cleaned_text) #finds the pattern in text for location and stores it
        location = location_match.group(0) if location_match else None

        # price + cuisine
        price_level = None #initializing new variables for price level and cuisine type
        cuisine = None

        price_cuisine_match = re.search(r"(\${1,4})\s*·\s*(.+)$", cleaned_text) #finds the patter for price and cuisine since they are listed next to each other on the website
        if price_cuisine_match:
            price_level = price_cuisine_match.group(1).strip() #stores the data for these variables group 1 is price level and group 2 is cuisine
            cuisine = price_cuisine_match.group(2).strip()

        # Michelin distinction
        michelin_distinction = None #initializing new variable for michelin ranking

        distinction_holder = card.find(attrs={"data-dtm-distinction": True}) #finds anything on the card to signify their respective rank
        if distinction_holder:
            michelin_distinction = distinction_holder.get("data-dtm-distinction") #if that value exists, stores it in this variable

        if not michelin_distinction: #if the rank isn't available on the card this loop uses the images on the card by accessing the file name to identify the rank
            for img in card.find_all("img"):
                src = (img.get("src") or "").lower()
                if "bib-gourmand" in src:
                    michelin_distinction = "Bib Gourmand"
                    break
                elif "green-star" in src:
                    michelin_distinction = "Green Star"
                    break
                elif "michelin-star" in src:
                    michelin_distinction = "Michelin Star"
                    break

        rows.append({
            "restaurant_name": restaurant_name,
            "restaurant_url": restaurant_url,
            "location": location,
            "price_level": price_level,
            "cuisine": cuisine,
            "michelin_distinction": michelin_distinction,
            "raw_card_text": cleaned_text,
            "page_number": page #adding page number in case I want to trace where each row came from
        }) #stores all restaurants data

    print(f"Rows scraped from page {page}:", len(rows))

    all_rows.extend(rows) #adds this page's rows to the full list of all pages

    page += 1 #move to the next Michelin results page
    time.sleep(2) #pause a little so I do not hammer the website too fast

print(f"\nStopped after page {page - 1}")
print("Total rows collected before cleaning:", len(all_rows))

#creating raw data frame from all scraped rows
df_michelin_raw = pd.DataFrame(all_rows)

print("Raw dataframe shape:", df_michelin_raw.shape)
print(df_michelin_raw.head())

#save raw scraped Michelin dataset
df_michelin_raw.to_csv("michelin_nyc_all_pages_raw.csv", index=False)
print("Saved as michelin_nyc_all_pages_raw.csv")


Scraping page 1: https://guide.michelin.com/us/en/new-york-state/new-york/restaurants
Status code: 200
Final URL: https://guide.michelin.com/us/en/new-york-state/new-york/restaurants
Content-Type: text/html;charset=UTF-8
Length: 1103772
Saved raw HTML as michelin_listing_raw_page_1.html
Number of cards found: 53
Rows scraped from page 1: 53

Scraping page 2: https://guide.michelin.com/us/en/new-york-state/new-york/restaurants/page/2
Status code: 200
Final URL: https://guide.michelin.com/us/en/new-york-state/new-york/restaurants/page/2
Content-Type: text/html;charset=UTF-8
Length: 1108923
Saved raw HTML as michelin_listing_raw_page_2.html
Number of cards found: 53
Rows scraped from page 2: 53

Scraping page 3: https://guide.michelin.com/us/en/new-york-state/new-york/restaurants/page/3
Status code: 200
Final URL: https://guide.michelin.com/us/en/new-york-state/new-york/restaurants/page/3
Content-Type: text/html;charset=UTF-8
Length: 1107371
Saved raw HTML as michelin_listing_raw_page_3.

In [65]:
#load raw Michelin scrape
df_michelin = pd.read_csv("michelin_nyc_all_pages_raw.csv")

#quick check before cleaning
print("Shape before cleaning:", df_michelin.shape)
print(df_michelin.head())

if df_michelin.empty:
    print("No rows were extracted.")
else:
    #removes rows with missing restaurant name or url because those are essential
    df_michelin = df_michelin.dropna(subset=["restaurant_name", "restaurant_url"])

    #drops duplicate restaurants based on Michelin url
    df_michelin = df_michelin.drop_duplicates(subset="restaurant_url").reset_index(drop=True)

    #function to clean restaurant name, location, price, and cuisine from raw card text
    def clean_card_fields(text):
        if pd.isna(text):
            return pd.Series([None, None, None, None])

        #collapse messy whitespace into single spaces
        text = re.sub(r"\s+", " ", str(text)).strip()

        #remove "Reserve a table | " only if it appears at the beginning
        text = re.sub(r"^Reserve a table \| ", "", text)

        #split the text into chunks using the | separator
        parts = [p.strip() for p in text.split("|")]

        #expected pattern after cleaning:
        #[restaurant_name, location, "$$$$ · Cuisine"]

        restaurant_name = None
        location = None
        price_level = None
        cuisine = None

        if len(parts) >= 1:
            restaurant_name = parts[0]

        if len(parts) >= 2:
            location = parts[1]

        if len(parts) >= 3:
            last_part = parts[2]

            #split price and cuisine on the dot separator
            pc_match = re.match(r"(\${1,4})\s*·\s*(.+)", last_part)
            if pc_match:
                price_level = pc_match.group(1).strip()
                cuisine = pc_match.group(2).strip()
            else:
                #fallback if separator is formatted strangely
                price_match = re.search(r"\${1,4}", last_part)
                if price_match:
                    price_level = price_match.group(0)
                    cuisine = last_part.replace(price_level, "").strip(" ·")

        return pd.Series([restaurant_name, location, price_level, cuisine])

    #apply the cleaning function to the raw card text column
    df_michelin[["restaurant_name_clean", "location_clean", "price_level_clean", "cuisine_clean"]] = (
        df_michelin["raw_card_text"].apply(clean_card_fields)
    )

    #overwrite original columns with cleaned values when available
    df_michelin["restaurant_name"] = df_michelin["restaurant_name_clean"].fillna(df_michelin["restaurant_name"])
    df_michelin["location"] = df_michelin["location_clean"].fillna(df_michelin["location"])
    df_michelin["price_level"] = df_michelin["price_level_clean"].fillna(df_michelin["price_level"])
    df_michelin["cuisine"] = df_michelin["cuisine_clean"].fillna(df_michelin["cuisine"])

    #clean Michelin distinction labels
    df_michelin["michelin_distinction"] = (
        df_michelin["michelin_distinction"]
        .replace("", pd.NA)
        .fillna("Selected Restaurants")
        .replace({
            "1 star": "One Star",
            "2 star": "Two Stars",
            "3 star": "Three Stars"
        })
    )

    #drop helper columns now that cleaning is done
    df_michelin = df_michelin.drop(columns=[
        "restaurant_name_clean",
        "location_clean",
        "price_level_clean",
        "cuisine_clean"
    ])

    #final duplicate check just in case
    df_michelin = df_michelin.drop_duplicates(subset="restaurant_url").reset_index(drop=True)

    #preview cleaned dataframe
    print("\nShape after cleaning:", df_michelin.shape)
    print(df_michelin[[
        "restaurant_name",
        "location",
        "price_level",
        "cuisine",
        "michelin_distinction",
        "page_number"
    ]].head(20))

    #save cleaned dataset
    df_michelin.to_csv("michelin_nyc_all_pages_clean.csv", index=False)
    print("\nSaved as michelin_nyc_all_pages_clean.csv")

Shape before cleaning: (453, 8)
  restaurant_name                                     restaurant_url  \
0            Icca  https://guide.michelin.com/us/en/new-york-stat...   
1            Icca  https://guide.michelin.com/us/en/new-york-stat...   
2        Chambers  https://guide.michelin.com/us/en/new-york-stat...   
3             Ito  https://guide.michelin.com/us/en/new-york-stat...   
4           Atera  https://guide.michelin.com/us/en/new-york-stat...   

             location price_level  \
0   New York, NY, USA        $$$$   
1   New York, NY, USA        $$$$   
2   New York, NY, USA         $$$   
3   New York, NY, USA        $$$$   
4   New York, NY, USA        $$$$   

                                             cuisine michelin_distinction  \
0  Japanese | Reserve a table | Chambers | New Yo...               1 star   
1                                           Japanese               1 star   
2                                       Contemporary                  NaN   
3   

In [66]:
print(df_michelin.info())
print(df_michelin.sample(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354 entries, 0 to 353
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   restaurant_name       354 non-null    object
 1   restaurant_url        354 non-null    object
 2   location              354 non-null    object
 3   price_level           354 non-null    object
 4   cuisine               354 non-null    object
 5   michelin_distinction  354 non-null    object
 6   raw_card_text         354 non-null    object
 7   page_number           354 non-null    int64 
dtypes: int64(1), object(7)
memory usage: 22.3+ KB
None
              restaurant_name  \
263                 The Grill   
180           Red Hook Tavern   
327                  Tanoreen   
140  Genesis House Restaurant   
38                    Raoul's   
191                   HanGawi   
150                   Massara   
172                    Upland   
14                       Saga   
88        